In [ ]:
# @title Setup
from google.cloud import bigquery
from google.colab import data_table
import bigframes.pandas as bpd

project = 'smiling-rhythm-486213-b9'
location = 'US'
client = bigquery.Client(project=project, location=location)
data_table.enable_dataframe_formatter()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Extract the current year
from datetime import datetime
current_year = datetime.now().year
current_year

In [ ]:
# Function to execute a BigQuery query and return a DataFrame

def query_to_dataframe(query: str) -> pd.DataFrame:
    """
    Executes a SQL query in BigQuery and returns a Pandas DataFrame.

    Parameters:
    - query (str): The SQL query to execute.

    Return:
    - pd.DataFrame : The DataFrame containing the results of the query.
    """
    try:
        df = client.query(query).to_dataframe()
        print(f"Query executed successfully. Retrieved {df.shape[0]} rows.")
        return df
    except Exception as e:
        print(f"Error executing query: {e}")
        return pd.DataFrame()

### Question 10: How often do passengers tip, and what factors (time of day, borough, fare amount) influence tip amounts?


In [ ]:
query_tipping_behavior_analysis = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.tipping_behavior_analysis`
"""
tipping_behavior_analysis_df = query_to_dataframe(query_tipping_behavior_analysis)
tipping_behavior_analysis_df.head()

In [ ]:
filtered_tipping_behavior_analysis_df = tipping_behavior_analysis_df[(tipping_behavior_analysis_df['year'] >= 2020) & (tipping_behavior_analysis_df['year'] <= current_year)]
filtered_tipping_behavior_analysis_df.info()

In [ ]:
filtered_tipping_behavior_analysis_df["trip_date"] = pd.to_datetime(filtered_tipping_behavior_analysis_df["trip_date"])
df_monthly = filtered_tipping_behavior_analysis_df.groupby(pd.Grouper(key="trip_date", freq="M"))["tip_frequency_percentage"].mean().reset_index()
df_monthly.head()

In [ ]:
fig = px.line(
    df_monthly,
    x="trip_date",
    y="tip_frequency_percentage",
    title="Tip Frequency Over Time (Monthly Average)",
    labels={"tip_frequency_percentage": "Tip Frequency (%)", "trip_date": "Date"},
    template="plotly_white",
    line_shape="spline"
)

fig.show()

In [ ]:
fig = px.bar(
    filtered_tipping_behavior_analysis_df.groupby("pickup_borough")["avg_tip_percentage"].mean().reset_index(),
    x="pickup_borough",
    y="avg_tip_percentage",
    title="Average Tip Percentage by Pickup Borough",
    labels={"avg_tip_percentage": "Average Tip (%)"},
    template="plotly_white",
    color="avg_tip_percentage",
    color_continuous_scale="Blues"
)
fig.show()

In [ ]:
heatmap_data = filtered_tipping_behavior_analysis_df.groupby(["pickup_hour", "pickup_borough"])["avg_tip_percentage"].mean().unstack()

fig = go.Figure(
    data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale="YlGnBu"
    )
)

fig.update_layout(
    title="Tipping Trends by Time of Day & Borough",
    xaxis_title="Borough",
    yaxis_title="Hour of Day",
    template="plotly_white"
)
fig.show()

### Question 11: How much revenue is generated from additional charges (MTA tax, congestion surcharge, airport fees), and has it changed over time?


In [ ]:
query_additional_charges_revenue = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.additional_charges_revenue`
"""
additional_charges_revenue_df = query_to_dataframe(query_additional_charges_revenue)
additional_charges_revenue_df.head()

In [ ]:
filtered_additional_charges_revenue_df = additional_charges_revenue_df[(additional_charges_revenue_df['year'] >= 2020) & (additional_charges_revenue_df['year'] <= current_year)]
filtered_additional_charges_revenue_df.info()

In [ ]:
filtered_additional_charges_revenue_df["trip_date"] = pd.to_datetime(filtered_additional_charges_revenue_df["trip_date"])
filtered_additional_charges_revenue_df.fillna(0, inplace=True)

fig = px.area(
    filtered_additional_charges_revenue_df,
    x="trip_date",
    y=["total_MTA_tax", "total_congestion_surcharge", "total_airport_fees"],
    labels={"trip_date": "Date", "value": "Revenue ($)", "variable": "Charge Type"},
    title="Revenue from Additional Charges Over Time",
    template="plotly_white"
)

fig.show()